# Pipeline 1 End-to-End Kaggle Notebook

This notebook turns the Pipeline 1 documentation into a single staged workflow that can run on Kaggle.

## Design goals
- Keep one global variable for the data root so it can be changed manually.
- Save all generated artifacts under `/kaggle/working`.
- Follow the documented pipeline order: data loading, EDA, preprocessing, splits, training, validation, checkpoint selection, inference, and optional ensemble.
- Stay runnable even if the local `pipeline_1/src` package is not available by falling back to notebook-local helpers.

## Execution plan
1. Resolve the competition data directory.
2. Load and validate train/test metadata.
3. Build simple EDA outputs and persist metadata.
4. Preprocess images and create stratified folds.
5. Train a fold-based classifier, save all checkpoints, and track metrics.
6. Select the best checkpoint per fold using generalization score.
7. Run fold ensemble inference and write `submission.csv` to `/kaggle/working`.

In [1]:
from __future__ import annotations

import json
import os
import random
import shutil
import sys
import glob
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
# TPU / XLA imports
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl


/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


In [2]:
# Global configuration that you can change manually
COMPETITION_ROOT = Path('/kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge')
DATA_ROOT = COMPETITION_ROOT / 'Data' / 'Data'
TRAIN_CSV = DATA_ROOT / 'training.csv'
TEST_CSV = DATA_ROOT / 'test.csv'
TRAIN_DIR = DATA_ROOT / 'Training'
TEST_DIR = DATA_ROOT / 'Test'
SOURCES = DATA_ROOT / 'sources.txt'
WORKING_ROOT = Path('/kaggle/working')
USE_MANIFEST = True
MANIFEST_DIR_PATH = '/kaggle/input/datasets/punyakdei/dlmmdd-pipe-1-material'  # paste the directory that contains a prior run's processed/checkpoints/outputs here
SEED = 42
NUM_CLASSES = 10
IMAGE_SIZE = 224
NUM_FOLDS = 5
BATCH_SIZE = 64
NUM_WORKERS = 2
PIN_MEMORY = True
PERSISTENT_WORKERS = True
NUM_EPOCHS = 20
LEARNING_RATE = 1.5e-4
WEIGHT_DECAY = 2e-4
DROPOUT_RATE = 0.4
DROPOUT_PATH_RATE = 0.3
MODEL_NAME = 'tf_efficientnetv2_l.in21k_ft_in1k'
USE_TTA = True
RUN_TRAINING = True  # set True when you want the full training loop to execute
CHECKPOINT_KEEP_TOP_K = 1
CHECKPOINT_SELECTION_METRIC = 'generalization_score'
RUN_INFERENCE_ONLY = False
INFERENCE_ONLY_PATH = '/kaggle/input/models/punyakdei/pipe1-general-models/pytorch/default/1'

WORKING_ROOT.mkdir(parents=True, exist_ok=True)

if RUN_INFERENCE_ONLY:
    RUN_TRAINING = False
    print("TRAINING turned OFF INFERENCE ONLY mode selected.")

# Detect accelerator: TPU > GPU > CPU
def get_device():
    try:
        import torch_xla.core.xla_model as xm
        device = xm.xla_device()
        return device, 'tpu'
    except Exception:
        pass
    if torch.cuda.is_available():
        return torch.device('cuda'), 'gpu'
    return torch.device('cpu'), 'cpu'

DEVICE, DEVICE_TYPE = get_device()

CPU_OPTIMIZED = False

if DEVICE_TYPE == 'tpu':
    NUM_WORKERS = 0
    PIN_MEMORY = False
    PERSISTENT_WORKERS = False
elif DEVICE_TYPE == 'cpu':
    NUM_WORKERS = 0
    PIN_MEMORY = False
    PERSISTENT_WORKERS = False
    CPU_OPTIMIZED = True

print(DEVICE)
if DEVICE_TYPE == 'tpu':
    print("Optimized for TPU")
elif CPU_OPTIMIZED:
    print("Optimized for CPU")
else:
    print("Optimized for GPU")

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE_TYPE == 'gpu':
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

xla:0
Optimized for TPU


/tmp/ipykernel_73/1728873037.py:43: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
E0000 00:00:1779438129.880277      73 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238


In [3]:
def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(obj, path: Path) -> None:
    ensure_dir(path.parent)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, default=str)


def safe_write_dataframe(df: pd.DataFrame, path: Path) -> None:
    ensure_dir(path.parent)
    if path.suffix == '.parquet':
        try:
            df.to_parquet(path, index=False)
            return
        except Exception:
            path = path.with_suffix('.csv')
    df.to_csv(path, index=False)


def manifest_path(*parts: str) -> Path:
    if not MANIFEST_DIR_PATH:
        raise ValueError('Set MANIFEST_DIR_PATH before using manifest mode.')
    return Path(MANIFEST_DIR_PATH).joinpath(*parts)


def load_manifest_artifact(path: Path):
    if not path.exists():
        raise FileNotFoundError(f'Manifest artifact not found: {path}')
    if path.suffix == '.npy':
        return np.load(path, allow_pickle=True)
    if path.suffix == '.json':
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    if path.suffix == '.parquet':
        try:
            return pd.read_parquet(path)
        except Exception:
            csv_path = path.with_suffix('.csv')
            if csv_path.exists():
                return pd.read_csv(csv_path)
            raise
    if path.suffix == '.csv':
        return pd.read_csv(path)
    return path


def load_manifest_bundle() -> dict:
    base = Path(MANIFEST_DIR_PATH)
    if not base.exists():
        raise FileNotFoundError(f'Manifest directory does not exist: {base}')

    bundle = {
        'train_meta': None,
        'test_meta': None,
        'X_train': None,
        'X_test': None,
        'y_train': None,
        'fold_metadata': None,
        'source_mapping': None,
    }

    candidates = {
        'train_meta': [base / 'processed' / 'train_metadata.parquet', base / 'processed' / 'train_metadata.csv'],
        'test_meta': [base / 'processed' / 'test_metadata.parquet', base / 'processed' / 'test_metadata.csv'],
        'X_train': [base / 'processed' / 'X_train.npy'],
        'X_test': [base / 'processed' / 'X_test.npy'],
        'y_train': [base / 'processed' / 'y_train.npy'],
        'fold_metadata': [base / 'processed' / 'fold_metadata.json'],
        'source_mapping': [base / 'processed' / 'source_mapping.json'],
    }

    for key, paths in candidates.items():
        for path in paths:
            if path.exists():
                bundle[key] = load_manifest_artifact(path)
                break

    missing = [key for key, value in bundle.items() if value is None and key in {'train_meta', 'test_meta', 'X_train', 'X_test', 'y_train', 'fold_metadata'}]
    if missing:
        raise FileNotFoundError(f'Manifest mode is missing required artifacts: {missing}')

    return bundle


PROJECT_ROOT = None
for candidate in [Path.cwd().resolve().parent / 'pipeline_1', Path.cwd().resolve() / 'pipeline_1', Path('/kaggle/input/pipeline_1'), Path('/kaggle/input/dlmmdd-workshop-synthetic-source-attribution-challenge/pipeline_1')]:
    if (candidate / 'src').exists():
        PROJECT_ROOT = candidate
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

PIPELINE_IMPORTS_AVAILABLE = False
if PROJECT_ROOT is not None:
    try:
        from src import DataLoader, ImagePreprocessor, TrainValSplitter, MetricsComputer, CheckpointManager, CheckpointSelector
        PIPELINE_IMPORTS_AVAILABLE = True
    except Exception as import_error:
        print(f'Pipeline package import fallback engaged: {import_error}')

DATA_DIR = DATA_ROOT
INPUT_TRAIN_CSV = TRAIN_CSV
INPUT_TEST_CSV = TEST_CSV
TRAIN_IMAGE_DIR = TRAIN_DIR
TEST_IMAGE_DIR = TEST_DIR
SOURCE_FILE = SOURCES
PROCESSED_DIR = ensure_dir(WORKING_ROOT / 'processed')
CHECKPOINT_DIR = ensure_dir(WORKING_ROOT / 'checkpoints')
FINAL_MODELS_DIR = ensure_dir(WORKING_ROOT / 'final_models')
LOG_DIR = ensure_dir(WORKING_ROOT / 'logs')
OUTPUT_DIR = ensure_dir(WORKING_ROOT / 'outputs')
EDA_DIR = ensure_dir(OUTPUT_DIR / 'eda')
EDA_PLOTS_DIR = ensure_dir(EDA_DIR / 'plots')
VALIDATION_DIR = ensure_dir(OUTPUT_DIR / 'validation')
INFERENCE_DIR = ensure_dir(OUTPUT_DIR / 'inference')

if RUN_INFERENCE_ONLY:
    FINAL_MODELS_DIR = Path(INFERENCE_ONLY_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Train CSV: {TRAIN_CSV}')
print(f'Test CSV: {TEST_CSV}')
print(f'Train dir: {TRAIN_DIR}')
print(f'Test dir: {TEST_DIR}')
print(f'Sources file: {SOURCES}')
print(f'Working directory: {WORKING_ROOT}')
print(f'Pipeline imports available: {PIPELINE_IMPORTS_AVAILABLE}')
print(f'Use manifest: {USE_MANIFEST}')
print(f'Manifest dir path: {MANIFEST_DIR_PATH or "<empty>"}')

Data directory: /kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/Data
Train CSV: /kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/Data/training.csv
Test CSV: /kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/Data/test.csv
Train dir: /kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/Data/Training
Test dir: /kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/Data/Test
Sources file: /kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/Data/sources.txt
Working directory: /kaggle/working
Pipeline imports available: False
Use manifest: True
Manifest dir path: /kaggle/input/datasets/punyakdei/dlmmdd-pipe-1-material


## Stage 00 - Data Loading, Validation, and EDA

This stage loads the raw CSVs, resolves image paths, extracts metadata, validates class balance, and saves a compact report for downstream stages.

In [4]:
if USE_MANIFEST:
    print('Manifest mode enabled: Stage 00 is skipped. Stage 02 will load prepared artifacts from MANIFEST_DIR_PATH.')
else:
    if PIPELINE_IMPORTS_AVAILABLE:
        loader = DataLoader(str(DATA_DIR))
        train_df, test_df = loader.load_metadata()
    else:
        train_df = pd.read_csv(INPUT_TRAIN_CSV)
        test_df = pd.read_csv(INPUT_TEST_CSV)

    train_df = train_df.copy()
    test_df = test_df.copy()

    def normalize_image_path(split_dir: Path, raw_path: str) -> str:
        raw_path = Path(str(raw_path))
        if raw_path.is_absolute() and raw_path.exists():
            return str(raw_path)

        candidate = split_dir / raw_path.name
        if candidate.exists():
            return str(candidate)

        return str(split_dir / raw_path.name)

    train_df['full_path'] = train_df['path'].apply(lambda p: normalize_image_path(TRAIN_IMAGE_DIR, p))
    test_df['full_path'] = test_df['path'].apply(lambda p: normalize_image_path(TEST_IMAGE_DIR, p))

    def inspect_image(path: str) -> dict:
        try:
            image = Image.open(path)
            width, height = image.size
            return {
                'width': width,
                'height': height,
                'format': image.format,
                'color_mode': image.mode,
                'file_size_bytes': os.path.getsize(path),
                'is_readable': True,
                'error': None,
            }
        except Exception as exc:
            return {
                'width': None,
                'height': None,
                'format': None,
                'color_mode': None,
                'file_size_bytes': os.path.getsize(path) if os.path.exists(path) else None,
                'is_readable': False,
                'error': str(exc),
            }

    def attach_metadata(df: pd.DataFrame) -> pd.DataFrame:
        records = [inspect_image(path) for path in tqdm(df['full_path'], desc='Reading image metadata')]
        return pd.concat([df.reset_index(drop=True), pd.DataFrame(records)], axis=1)

    if PIPELINE_IMPORTS_AVAILABLE:
        train_meta = loader.extract_image_metadata(train_df)
        test_meta = loader.extract_image_metadata(test_df)
        source_mapping = loader.load_source_mapping()
        train_meta = loader.add_source_names(train_meta, source_mapping)
        validation_report = loader.validate_training_data(train_meta)
        data_stats = loader.compute_statistics(train_meta, test_meta)
    else:
        train_meta = attach_metadata(train_df)
        test_meta = attach_metadata(test_df)
        source_mapping = {}
        validation_report = {
            'is_valid': bool(train_meta['is_readable'].all()),
            'warnings': [],
            'errors': []
        }
        if 'y' in train_meta.columns:
            counts = train_meta['y'].value_counts().sort_index()
            validation_report['class_counts'] = counts.to_dict()
            if len(counts) != NUM_CLASSES:
                validation_report['warnings'].append(f'Expected {NUM_CLASSES} classes, found {len(counts)}')
        data_stats = {
            'train': {
                'total_samples': int(len(train_meta)),
                'classes': int(train_meta['y'].nunique()) if 'y' in train_meta.columns else None,
                'class_distribution': train_meta['y'].value_counts().sort_index().to_dict() if 'y' in train_meta.columns else {},
                'avg_height': float(train_meta['height'].mean()),
                'avg_width': float(train_meta['width'].mean()),
                'avg_file_size_mb': float(train_meta['file_size_bytes'].mean() / 1e6),
                'formats': train_meta['format'].value_counts().to_dict(),
                'color_modes': train_meta['color_mode'].value_counts().to_dict(),
            },
            'test': {
                'total_samples': int(len(test_meta)),
                'avg_height': float(test_meta['height'].mean()),
                'avg_width': float(test_meta['width'].mean()),
                'avg_file_size_mb': float(test_meta['file_size_bytes'].mean() / 1e6),
            },
        }

    class_counts = train_meta['y'].value_counts().sort_index()
    print(train_meta.head(3))
    print(train_meta.shape, test_meta.shape)
    print(class_counts)

    safe_write_dataframe(train_meta, PROCESSED_DIR / 'train_metadata.parquet')
    safe_write_dataframe(test_meta, PROCESSED_DIR / 'test_metadata.parquet')
    save_json(data_stats, EDA_DIR / 'data_stats.json')
    save_json(validation_report, EDA_DIR / 'data_validation_report.json')
    save_json(source_mapping, PROCESSED_DIR / 'source_mapping.json')

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.countplot(x='y', data=train_meta, ax=axes[0], color='#2a6fdb')
    axes[0].set_title('Training Class Distribution')
    axes[0].set_xlabel('Class ID')
    axes[0].set_ylabel('Count')

    sns.histplot(train_meta['width'].dropna(), kde=True, ax=axes[1], color='#d95f02')
    axes[1].set_title('Image Width Distribution')
    axes[1].set_xlabel('Width')
    plt.tight_layout()
    plt.savefig(EDA_PLOTS_DIR / 'eda_overview.png', dpi=160, bbox_inches='tight')
    plt.show()

Manifest mode enabled: Stage 00 is skipped. Stage 02 will load prepared artifacts from MANIFEST_DIR_PATH.


## Stage 01 - Preprocessing, Augmentation, and Stratified Folds

This stage caches resized image arrays for inspection, but the training loop below uses on-the-fly transforms so augmentation stays practical and memory-friendly.

In [5]:
if USE_MANIFEST:
    print('Manifest mode enabled: Stage 01 is skipped. Stage 02 will load X_train, X_test, y_train, and fold metadata from MANIFEST_DIR_PATH.')
else:
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]

    class NotebookImagePreprocessor:
        def __init__(self, target_size: int = 224):
            self.transform = transforms.Compose([
                transforms.Resize((target_size, target_size)),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ])

        def __call__(self, image_path: str) -> np.ndarray:
            image = Image.open(image_path).convert('RGB')
            return self.transform(image).numpy().astype(np.float32)

    preprocessor = NotebookImagePreprocessor(IMAGE_SIZE)

    def preprocess_paths(paths: pd.Series, cache_path: Path) -> np.ndarray:
        if cache_path.exists():
            return np.load(cache_path)
        batches = []
        for image_path in tqdm(paths, desc=f'Preprocessing {cache_path.stem}'):
            batches.append(preprocessor(image_path))
        array = np.stack(batches, axis=0)
        np.save(cache_path, array)
        return array

    X_train = preprocess_paths(train_meta['full_path'], PROCESSED_DIR / 'X_train.npy')
    X_test = preprocess_paths(test_meta['full_path'], PROCESSED_DIR / 'X_test.npy')
    y_train = train_meta['y'].to_numpy(dtype=np.int64)
    np.save(PROCESSED_DIR / 'y_train.npy', y_train)

    skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
    fold_metadata = {}
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y_train)), y_train)):
        fold_metadata[fold_idx] = {
            'fold_idx': int(fold_idx),
            'train_indices': train_idx.tolist(),
            'val_indices': val_idx.tolist(),
            'train_count': int(len(train_idx)),
            'val_count': int(len(val_idx)),
            'train_class_counts': np.bincount(y_train[train_idx], minlength=NUM_CLASSES).tolist(),
            'val_class_counts': np.bincount(y_train[val_idx], minlength=NUM_CLASSES).tolist(),
        }

    save_json(fold_metadata, PROCESSED_DIR / 'fold_metadata.json')
    print(X_train.shape, X_train.dtype)
    print(X_test.shape, X_test.dtype)
    print(fold_metadata[0])

Manifest mode enabled: Stage 01 is skipped. Stage 02 will load X_train, X_test, y_train, and fold metadata from MANIFEST_DIR_PATH.


## Stage 02 and 03 - Training, Validation, and Checkpoint Selection

The notebook trains one model per fold, saves every epoch checkpoint, computes validation metrics, and chooses the best epoch by generalization score.

In [6]:
try:
    import timm
    TIMM_AVAILABLE = True
except Exception:
    TIMM_AVAILABLE = False

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def restore_full_path_column(df: pd.DataFrame, split_dir: Path) -> pd.DataFrame:
    df = df.copy()
    if 'full_path' not in df.columns:
        df['full_path'] = df['path'].apply(lambda p: str(split_dir / Path(str(p)).name))
    return df


def get_selection_metric_value(metrics: dict) -> float:
    """Return the scalar used to rank checkpoints.

    Supported CHECKPOINT_SELECTION_METRIC values:
      "val_accuracy"         -> raw validation accuracy
      "generalization_score" -> val_acc penalised by train/val gap (default)
    Falls back to accuracy if generalization_score key is absent.
    """
    if CHECKPOINT_SELECTION_METRIC == 'val_accuracy':
        return float(metrics.get('accuracy', 0.0))
    value = metrics.get('generalization_score', metrics.get('accuracy', 0.0))
    return float(value)


def prune_to_top_k_checkpoints(saved_checkpoints: list, keep_top_k: int) -> list:
    """Delete on-disk checkpoint files outside the top-k by selection_value."""
    while len(saved_checkpoints) > keep_top_k:
        worst_idx = min(
            range(len(saved_checkpoints)),
            key=lambda idx: saved_checkpoints[idx]['selection_value'],
        )
        worst = saved_checkpoints.pop(worst_idx)
        worst_path = Path(worst['checkpoint_path'])
        if worst_path.exists():
            worst_path.unlink()
            print(f"  [prune] deleted {worst_path.name} (selection_value={worst['selection_value']:.4f})")
    return saved_checkpoints


# ── manifest / non-manifest data loading ─────────────────────────────────────
if USE_MANIFEST:
    print("Loading Stage 01 artifacts from manifest ...")
    manifest_bundle = load_manifest_bundle()
    train_meta = restore_full_path_column(manifest_bundle['train_meta'], TRAIN_IMAGE_DIR)
    test_meta  = restore_full_path_column(manifest_bundle['test_meta'],  TEST_IMAGE_DIR)
    # X_train / X_test are loaded for manifest compatibility but the training
    # loop reads images on-the-fly from full_path, so these arrays are not used.
    X_train = manifest_bundle['X_train']
    X_test  = manifest_bundle['X_test']
    y_train = np.asarray(manifest_bundle['y_train'], dtype=np.int64)
    loaded_fold_metadata = manifest_bundle['fold_metadata']
    if isinstance(loaded_fold_metadata, dict):
        fold_metadata = {int(key): value for key, value in loaded_fold_metadata.items()}
    else:
        fold_metadata = loaded_fold_metadata
    source_mapping = manifest_bundle.get('source_mapping') or {}
    print(f"  train_meta : {train_meta.shape}")
    print(f"  test_meta  : {test_meta.shape}")
    print(f"  y_train    : {y_train.shape}  classes={np.unique(y_train).tolist()}")
    print(f"  folds      : {len(fold_metadata)}")
    print("Manifest loaded successfully.")
else:
    y_train = train_meta['y'].to_numpy(dtype=np.int64)


# FIX: import TorchDataLoader under an unambiguous alias so that any future
# "from src import DataLoader" cannot silently shadow the PyTorch one.
from torch.utils.data import DataLoader as TorchDataLoader


def to_cpu(tensor):
    """Copy a tensor to CPU, flushing the XLA graph first on TPU.

    On GPU: equivalent to tensor.cpu().
    On TPU/XLA: calls xm.mark_step() to flush pending ops before the
    host copy, preventing the "can't convert xla device tensor to numpy"
    TypeError that surfaces during both training metric collection and
    inference id extraction.
    """
    if DEVICE_TYPE == 'tpu':
        xm.mark_step()
    return tensor.cpu()


class NumpyImageDataset(Dataset):
    """On-the-fly image loader for train and validation splits."""

    def __init__(self, df: pd.DataFrame, indices, train: bool = True):
        self.df    = df.iloc[list(indices)].reset_index(drop=True)
        self.train = train
        if train:
            self.transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(15),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['full_path']).convert('RGB')
        image = self.transform(image)
        label = int(row['y']) if 'y' in row and not pd.isna(row['y']) else -1
        return image, label


def build_model(model_name: str, num_classes: int):
    """Build the classifier backbone. Prefers timm; falls back to torchvision."""
    if TIMM_AVAILABLE:
        print(f"  [model] building '{model_name}' via timm")
        return timm.create_model(
            model_name,
            pretrained=True,
            num_classes=num_classes,
            drop_rate=DROPOUT_RATE,
            drop_path_rate=DROPOUT_PATH_RATE,
        )
    print("  [model] timm not available -- falling back to torchvision")
    if model_name.startswith('efficientnet'):
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=DROPOUT_RATE, inplace=True),
            nn.Linear(in_features, num_classes),
        )
        return model
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def compute_epoch_metrics(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    }


def generalization_score(train_metrics, val_metrics, alpha=4.0):
    """Penalise val_acc by the squared train/val accuracy gap.

    Formula: gen = val_acc - alpha * max(0, train_acc - val_acc)^2

    With alpha=4.0 the penalty only becomes notable (>0.01) when the gap
    exceeds 0.05 (5 pp). At a 0.30 gap the penalty is 0.36, which can push
    the score below zero -- this is intentional: it prevents heavily overfit
    epochs from being selected as best checkpoints.
    """
    train_acc = train_metrics.get('accuracy', 0.0)
    val_acc   = val_metrics.get('accuracy', 0.0)
    gap       = max(0.0, train_acc - val_acc)
    return float(val_acc - alpha * (gap ** 2))


def evaluate_model(model, loader, criterion):
    """Full validation pass. Returns (metrics_dict, y_true_array, y_pred_array).

    FIX: all tensor->numpy conversions now go through to_cpu() so this
    function works correctly on both GPU and TPU/XLA without hanging or
    raising TypeError.
    """
    model.eval()
    losses = []
    y_true = []
    y_pred = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            loss   = criterion(logits, labels)
            losses.append(loss.item())
            preds  = logits.argmax(dim=1)
            # FIX: flush XLA computation graph before copying to host
            y_true.extend(to_cpu(labels).numpy().tolist())
            y_pred.extend(to_cpu(preds).numpy().tolist())
    metrics         = compute_epoch_metrics(np.array(y_true), np.array(y_pred))
    metrics['loss'] = float(np.mean(losses)) if losses else None
    return metrics, np.array(y_true), np.array(y_pred)


def train_fold(fold_idx: int, fold_info: dict, train_df: pd.DataFrame):
    """Train one fold end-to-end, checkpoint every epoch, keep top-k."""
    print(f"\n{'='*64}")
    print(f"  FOLD {fold_idx}  |  train={fold_info['train_count']}  val={fold_info['val_count']}")
    print(f"{'='*64}")

    fold_dir          = ensure_dir(CHECKPOINT_DIR / f'fold_{fold_idx}')
    history           = []
    saved_checkpoints = []

    train_dataset = NumpyImageDataset(train_df, fold_info['train_indices'], train=True)
    val_dataset   = NumpyImageDataset(train_df, fold_info['val_indices'],   train=False)

    # FIX: use TorchDataLoader alias so it cannot be shadowed by src imports
    train_loader = TorchDataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    val_loader = TorchDataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    if DEVICE_TYPE == 'tpu':
        train_loader = pl.MpDeviceLoader(train_loader, DEVICE)
        val_loader   = pl.MpDeviceLoader(val_loader,   DEVICE)

    model     = build_model(MODEL_NAME, NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(NUM_EPOCHS):
        # ── training pass ─────────────────────────────────────────────────
        model.train()
        train_losses = []
        y_true_train = []
        y_pred_train = []

        for images, labels in tqdm(
            train_loader,
            desc=f'Fold {fold_idx} Epoch {epoch + 1}/{NUM_EPOCHS}',
            leave=False,
        ):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss   = criterion(logits, labels)
            loss.backward()
            if DEVICE_TYPE == 'tpu':
                xm.optimizer_step(optimizer)
            else:
                optimizer.step()
            train_losses.append(loss.item())
            preds = logits.argmax(dim=1)
            # FIX: flush XLA graph before numpy conversion
            y_true_train.extend(to_cpu(labels).numpy().tolist())
            y_pred_train.extend(to_cpu(preds).numpy().tolist())

        train_metrics         = compute_epoch_metrics(np.array(y_true_train), np.array(y_pred_train))
        train_metrics['loss'] = float(np.mean(train_losses)) if train_losses else None

        # ── validation pass ───────────────────────────────────────────────
        val_metrics, _, _                   = evaluate_model(model, val_loader, criterion)
        val_metrics['generalization_score'] = generalization_score(train_metrics, val_metrics)
        current_lr      = optimizer.param_groups[0]['lr']
        selection_value = get_selection_metric_value(val_metrics)

        # ── checkpoint ────────────────────────────────────────────────────
        checkpoint = {
            'fold_idx':             fold_idx,
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_metrics':        train_metrics,
            'val_metrics':          val_metrics,
            'learning_rate':        current_lr,
            'model_name':           MODEL_NAME,
        }
        checkpoint_path = fold_dir / f'epoch_{epoch:03d}.pth'
        if DEVICE_TYPE == 'tpu':
            xm.save(checkpoint, checkpoint_path)
        else:
            torch.save(checkpoint, checkpoint_path)

        saved_checkpoints.append({
            'epoch':           epoch,
            'checkpoint_path': str(checkpoint_path),
            'selection_value': selection_value,
        })
        saved_checkpoints = prune_to_top_k_checkpoints(saved_checkpoints, CHECKPOINT_KEEP_TOP_K)

        history.append({
            'epoch':           epoch,
            'checkpoint_path': str(checkpoint_path),
            'train':           train_metrics,
            'val':             val_metrics,
            'learning_rate':   current_lr,
            'selection_value': selection_value,
        })
        scheduler.step()

        gap_pp = (train_metrics['accuracy'] - val_metrics['accuracy']) * 100
        print(
            f"Fold {fold_idx} Epoch {epoch:03d} | "
            f"train_acc={train_metrics['accuracy']:.4f}  "
            f"val_acc={val_metrics['accuracy']:.4f}  "
            f"val_f1={val_metrics['f1_macro']:.4f}  "
            f"gen={val_metrics['generalization_score']:.4f}  "
            f"gap={gap_pp:+.1f}pp  "
            f"lr={current_lr:.2e}"
        )

    save_json(history,           LOG_DIR / f'training_history_fold_{fold_idx}.json')
    save_json(saved_checkpoints, LOG_DIR / f'checkpoint_retention_fold_{fold_idx}.json')

    best = max(history, key=lambda r: r['selection_value'])
    print(
        f"\n  Fold {fold_idx} complete. "
        f"Best epoch={best['epoch']:03d}  "
        f"val_acc={best['val']['accuracy']:.4f}  "
        f"gen={best['val']['generalization_score']:.4f}"
    )
    return history


# ── training loop ─────────────────────────────────────────────────────────────
training_histories = {}
if RUN_TRAINING:
    print(f"\nStarting {NUM_FOLDS}-fold training ... (epochs={NUM_EPOCHS}, model={MODEL_NAME})")
    for fold_idx in range(NUM_FOLDS):
        training_histories[fold_idx] = train_fold(fold_idx, fold_metadata[fold_idx], train_meta)
    print("\nAll folds complete.")
else:
    print('RUN_TRAINING is False -- training loop skipped. Set RUN_TRAINING=True to train.')


# ── checkpoint selection ───────────────────────────────────────────────────────
selection_rows   = []
best_checkpoints = {}

for fold_idx, history in training_histories.items():
    if not history:
        continue
    best_row = max(history, key=lambda item: item['selection_value'])
    best_checkpoints[fold_idx] = best_row
    selection_rows.append({
        'fold_idx':             fold_idx,
        'best_epoch':           best_row['epoch'],
        'checkpoint_path':      best_row['checkpoint_path'],
        'train_accuracy':       best_row['train']['accuracy'],
        'val_accuracy':         best_row['val']['accuracy'],
        'val_f1_macro':         best_row['val']['f1_macro'],
        'generalization_score': best_row['val']['generalization_score'],
        'selection_value':      best_row['selection_value'],
    })

selection_df = pd.DataFrame(selection_rows)
if not selection_df.empty:
    selection_df.to_csv(VALIDATION_DIR / 'validation_metrics.csv', index=False)
    save_json(selection_rows, VALIDATION_DIR / 'checkpoint_selection.json')
    print("\nCopying best checkpoints to final_models/ ...")
    for row in selection_rows:
        fold_dir = CHECKPOINT_DIR / f"fold_{row['fold_idx']}"
        src      = fold_dir / f"epoch_{row['best_epoch']:03d}.pth"
        dst      = FINAL_MODELS_DIR / f"fold_{row['fold_idx']}_best.pth"
        if src.exists():
            shutil.copy2(src, dst)
            print(
                f"  [select] fold {row['fold_idx']} -> epoch {row['best_epoch']:03d}  "
                f"val_acc={row['val_accuracy']:.4f}  "
                f"gen={row['generalization_score']:.4f}"
            )
    print("\nCheckpoint selection summary:")
    print(selection_df.to_string(index=False))
else:
    print('No trained histories available -- checkpoint selection has not run.')


Manifest mode enabled: loaded Stage 01 artifacts and fold metadata from MANIFEST_DIR_PATH.


model.safetensors:   0%|          | 0.00/476M [00:00<?, ?B/s]

Fold 0 Epoch 1/20:   0%|          | 0/88 [00:00<?, ?it/s]

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Fold 0 Epoch 000 | train_acc=0.4198 | val_acc=0.7850 | gen=0.7850


...

Fold 4 Epoch 19/20:   0%|          | 0/88 [00:00<?, ?it/s]

Fold 4 Epoch 018 | train_acc=0.9943 | val_acc=0.9679 | gen=0.9651


Fold 4 Epoch 20/20:   0%|          | 0/88 [00:00<?, ?it/s]

Fold 4 Epoch 019 | train_acc=0.9946 | val_acc=0.9671 | gen=0.9641


   fold_idx  best_epoch                                   checkpoint_path  \
0         0           8  /kaggle/working/checkpoints/fold_0/epoch_008.pth   
1         1          11  /kaggle/working/checkpoints/fold_1/epoch_011.pth   
2         2           8  /kaggle/working/checkpoints/fold_2/epoch_008.pth   
3         3          17  /kaggle/working/checkpoints/fold_3/epoch_017.pth   
4         4          14  /kaggle/working/checkpoints/fold_4/epoch_014.pth   

   train_accuracy  val_accuracy  val_f1_macro  generalization_score  \
0        0.975893      0.960000      0.960042              0.958990   
1        0.987679      0.975000      0.975004              0.974357   
2        0.977143      0.959286      0.959169              0.958010   
3        0.993214      0.977857      0.977853              0.976914   
4        0.991786      0.970714      0.970706              0.968938   

   selection_value  
0         0.958990  
1         0.974357  
2         0.958010  
3         0.976914  
4    

## Stage 04 and 05 - Inference, Submission, and Optional Ensemble

This stage loads the selected fold checkpoints, averages the predicted probabilities, writes the Kaggle submission file, and saves confidence diagnostics.

In [7]:
class TestImageDataset(Dataset):
    """Dataset wrapper for the test split (no labels)."""

    def __init__(self, df: pd.DataFrame):
        self.df        = df.reset_index(drop=True)
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['full_path']).convert('RGB')
        return self.transform(image), int(row['ID'])


def load_checkpoint_model(checkpoint_path: Path):
    """Load a saved .pth checkpoint and return (model, raw_checkpoint_dict)."""
    print(f"  [load] {checkpoint_path.name}")
    checkpoint       = torch.load(checkpoint_path, map_location='cpu')
    saved_model_name = checkpoint.get('model_name', MODEL_NAME)
    model = build_model(saved_model_name, NUM_CLASSES)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(DEVICE)
    model.eval()
    saved_val = checkpoint.get('val_metrics', {})
    print(f"         architecture : {saved_model_name}")
    print(f"         epoch        : {checkpoint.get('epoch', '?')}")
    print(f"         val_acc      : {saved_val.get('accuracy', float('nan')):.4f}")
    print(f"         gen_score    : {saved_val.get('generalization_score', float('nan')):.4f}")
    return model, checkpoint


def predict_probabilities(model, loader):
    """Run inference and return (probs_array, ids_array) both as numpy.

    FIX: ids tensor was converted to numpy directly on the XLA device, causing
    "can't convert xla:0 device type tensor to numpy" TypeError. Both probs
    and ids now go through to_cpu() before .numpy().
    """
    probabilities = []
    identifiers   = []

    with torch.no_grad():
        for images, ids in tqdm(loader, desc='Inference', leave=False):
            images   = images.to(DEVICE)
            logits   = model(images)
            probs    = torch.softmax(logits, dim=1)
            # FIX: flush XLA graph before host copy via to_cpu()
            probs_np = to_cpu(probs).numpy()
            probabilities.append(probs_np)
            # FIX: MpDeviceLoader wraps ids in XLA tensors; to_cpu() needed here too
            identifiers.extend(to_cpu(ids).numpy().tolist())

    print(f"  [infer] {len(identifiers)} predictions collected")
    return np.concatenate(probabilities, axis=0), np.array(identifiers)


if not test_meta.empty:
    print(f"\nRunning inference on {len(test_meta)} test images ...")

    test_dataset = TestImageDataset(test_meta)
    # FIX: use TorchDataLoader alias (consistent with train_fold fix)
    test_loader = TorchDataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    if DEVICE_TYPE == 'tpu':
        test_loader = pl.MpDeviceLoader(test_loader, DEVICE)

    candidate_model_paths = sorted(FINAL_MODELS_DIR.glob('fold_*_best.pth'))
    print(f"Found {len(candidate_model_paths)} fold checkpoint(s): "
          f"{[p.name for p in candidate_model_paths]}")

    all_fold_probs = []
    fold_weights   = []

    for checkpoint_path in candidate_model_paths:
        model, checkpoint = load_checkpoint_model(checkpoint_path)
        fold_probs, ordered_ids = predict_probabilities(model, test_loader)
        all_fold_probs.append(fold_probs)
        w = checkpoint.get('val_metrics', {}).get('accuracy', 1.0)
        fold_weights.append(w)
        print(f"  [weight] {checkpoint_path.name}  val_acc_weight={w:.4f}")

    if all_fold_probs:
        all_fold_probs = np.stack(all_fold_probs, axis=0)  # (n_folds, n_test, n_classes)
        fold_weights   = np.asarray(fold_weights, dtype=np.float32)

        if np.isclose(fold_weights.sum(), 0):
            print("  [ensemble] all weights zero -- falling back to simple average")
            fold_weights   = None
            ensemble_probs = all_fold_probs.mean(axis=0)
        else:
            fold_weights   = fold_weights / fold_weights.sum()
            ensemble_probs = np.average(all_fold_probs, axis=0, weights=fold_weights)
            print(f"  [ensemble] weighted average  weights={fold_weights.round(4).tolist()}")

        ensemble_preds      = ensemble_probs.argmax(axis=1)
        ensemble_confidence = ensemble_probs.max(axis=1)

        # ── submission ───────────────────────────────────────────────────
        submission_df = test_meta[['ID']].copy()
        submission_df['TARGET'] = ensemble_preds.astype(int)
        submission_df.to_csv(WORKING_ROOT  / 'submission.csv', index=False)
        submission_df.to_csv(INFERENCE_DIR / 'submission.csv', index=False)

        confidence_df = pd.DataFrame({
            'ID':              test_meta['ID'].values,
            'predicted_class': ensemble_preds.astype(int),
            'confidence':      ensemble_confidence.astype(float),
        })
        confidence_df.to_csv(INFERENCE_DIR / 'prediction_confidence.csv', index=False)

        inference_metadata = {
            'timestamp':       datetime.utcnow().isoformat(),
            'num_models':      int(len(candidate_model_paths)),
            'ensemble_type':   'weighted_average' if fold_weights is not None else 'simple_average',
            'mean_confidence': float(ensemble_confidence.mean()),
            'min_confidence':  float(ensemble_confidence.min()),
            'max_confidence':  float(ensemble_confidence.max()),
            'num_predictions': int(len(ensemble_preds)),
        }
        save_json(inference_metadata, INFERENCE_DIR / 'submission_metadata.json')

        print(f"\nSubmission preview:")
        print(submission_df.head(10).to_string(index=False))
        print(f"\nSubmission saved to : {WORKING_ROOT / 'submission.csv'}")
        print(f"Average confidence  : {ensemble_confidence.mean():.4f}")
        print(f"Min / Max confidence: {ensemble_confidence.min():.4f} / {ensemble_confidence.max():.4f}")

        pred_counts = pd.Series(ensemble_preds).value_counts().sort_index()
        print(f"\nPrediction distribution (class -> count):")
        print(pred_counts.to_string())
    else:
        print('No fold checkpoints were found -- inference was skipped.')
else:
    print('test_meta is empty -- inference was skipped.')

# Optional ensemble note: if you later train multiple architectures, repeat
# the inference block above and combine probability tensors across model
# families before argmax.


Loaded checkpoint: fold_0_best.pth
Architecture: tf_efficientnetv2_l.in21k_ft_in1k


Inference:   0%|          | 0/47 [00:00<?, ?it/s]

TypeError: can't convert xla:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

# Manifest